# Session 5 — Capstone: Ask Your Own Question
### Brain MRI Tumour Segmentation · CS Academy Seminar

---

You have a working model and — since Session 4 — a way of measuring it that doesn't lie.

Today you use it to answer a question nobody has told you the answer to.

**Pick ONE track.** Spend ~80 minutes. Produce **one plot and three sentences**, then present for
five minutes.

| | track | the question | difficulty |
|---|---|---|---|
| **A** | Annotation budget | How many patients do you actually need? | approachable |
| **B** | The metric's blind spot | Is Dice fair to small tumours? | medium |
| **C** | Shape and genetics | Can tumour shape predict its DNA? | ambitious |

A null result is a real result. "I tested this and found nothing" — with evidence — is a
legitimate and respectable outcome. Do not fudge anything to get a prettier plot.

⏱ Roughly 2 hours including presentations.

In [ ]:
#@title Setup — run this first  { display-mode: "form" }
# Downloads the seminar helper code and the dataset.
REPO_RAW = "https://raw.githubusercontent.com/OTMAN-REPO/brain-mri-seminar/main"  #@param {type:"string"}
DATA_URL = ""  #@param {type:"string"}

import os, urllib.request
if not os.path.exists("seminar.py"):
    try:
        urllib.request.urlretrieve(f"{REPO_RAW}/seminar.py", "seminar.py")
        print("Got seminar.py")
    except Exception as e:
        raise SystemExit(f"Could not fetch seminar.py from {REPO_RAW}\n"
                         f"Upload it manually to this Colab session (folder icon on the left).\n{e}")

from seminar import *
import numpy as np, matplotlib.pyplot as plt
images, masks, patient_ids, slice_index = get_data(url=DATA_URL)
print(f"\n{len(images)} slices | {len(np.unique(patient_ids))} patients | image {images.shape[1:]}")
print("device:", DEVICE)

In [ ]:
import torch, time
assert DEVICE == "cuda", "Runtime -> Change runtime type -> T4 GPU"

def train_model(train_idx, val_idx, epochs=12, f=16, bs=16, augment=True, seed=0, verbose=False):
    torch.manual_seed(seed); np.random.seed(seed)
    model = UNet(f=f).to(DEVICE)
    dl = torch.utils.data.DataLoader(SliceDataset(images, masks, train_idx, augment=augment),
                                     batch_size=bs, shuffle=True, num_workers=2, drop_last=True)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    for ep in range(epochs):
        model.train()
        for x, y in dl:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(); combo_loss(model(x), y).backward(); opt.step()
        sched.step()
        if verbose: print(f"  ep {ep+1}", end="\r")
    return model, dice_score(predict_all(model, images, val_idx), masks[val_idx])

# ALWAYS split by patient now. You know why.
train_idx, val_idx = split_by_patient(patient_ids, val_frac=0.25, seed=0)
print(f"{len(np.unique(patient_ids[train_idx]))} train patients, "
      f"{len(np.unique(patient_ids[val_idx]))} val patients")

---
# Track A — How many patients do you actually need?

**Why anyone cares.** Every labelled scan in this dataset cost a radiologist their time. A hospital
starting a new project has to decide: do we pay for 20 annotated cases, or 200? Nobody can answer
that from theory. You can answer it from data.

**The experiment.** Train on 5, 10, 20, 40, then all available training patients. Validate on the
*same held-out patients* every time, so the only thing changing is training set size. Plot Dice
against number of patients.

**What to look for.** Does it keep climbing, or does it flatten? If it flattens, *where*? That elbow
is the answer to a real budgeting question.

⏱ ~5 training runs.

In [ ]:
#@title Track A
train_pats = np.unique(patient_ids[train_idx])
sizes = [5, 10, 20, 40, len(train_pats)]
sizes = sorted(set(s for s in sizes if s <= len(train_pats)))

curve = []
for n in sizes:
    keep = set(train_pats[:n])
    sub = np.array([i for i in train_idx if patient_ids[i] in keep])
    t0 = time.time()
    _, d = train_model(sub, val_idx, epochs=12, seed=0)
    curve.append((n, len(sub), d))
    print(f"{n:3d} patients ({len(sub):4d} slices) -> Dice {d:.4f}   [{time.time()-t0:.0f}s]")

c = np.array(curve, float)
plt.figure(figsize=(6,3.6))
plt.plot(c[:,0], c[:,2], "o-", ms=6)
plt.axhline(0.84, ls="--", c="crimson", lw=1, label="human agreement ~0.84")
plt.xlabel("training patients"); plt.ylabel("validation Dice")
plt.title("How much data do you actually need?"); plt.legend(fontsize=8); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()

# TODO: where is the elbow? If a hospital could only afford to annotate N cases,
#       what N would you recommend, and what does it cost you in Dice?

### Track A — going further
- Run each size with 2–3 different random *subsets* of patients and show error bars. With few
  patients, *which* patients you got matters enormously.
- Plot against **slices** instead of patients. Which is the better predictor of performance?
- Does augmentation matter more when data is scarce? (Run the small sizes with `augment=False`.)

---
# Track B — Is Dice fair to small tumours?

**Why anyone cares.** Every paper on this task reports a single averaged Dice. But Dice is a ratio,
and ratios behave badly when the denominator is small. If a tumour is 30 pixels and you miss the
boundary by one pixel all the way round, you lose a big fraction of your score. Do that on a
3000-pixel tumour and you barely notice.

If that's true, then **a model's headline Dice depends partly on how big the tumours in the test set
happen to be** — which means two papers reporting different numbers might have equally good models.

**The experiment.** Score every validation slice individually. Plot Dice against tumour area.

**What to look for.** Is there a relationship? Where does it break down? Is there a size below which
the model is essentially hopeless?

In [ ]:
#@title Track B
model, overall = train_model(train_idx, val_idx, epochs=20, seed=0)
pred = predict_all(model, images, val_idx)
print(f"overall Dice: {overall:.4f}")

areas, dices = [], []
for k in range(len(val_idx)):
    a = int(masks[val_idx[k]].sum())
    if a > 0:
        areas.append(a); dices.append(dice_score(pred[k], masks[val_idx[k]]))
areas, dices = np.array(areas), np.array(dices)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].scatter(areas, dices, s=14, alpha=.5)
ax[0].set_xscale("log"); ax[0].set_xlabel("tumour area (pixels, log)"); ax[0].set_ylabel("Dice")
ax[0].set_title("per-slice Dice vs tumour size"); ax[0].grid(alpha=.3)

bins = np.percentile(areas, [0,20,40,60,80,100])
mids, means = [], []
for i in range(5):
    s = (areas >= bins[i]) & (areas <= bins[i+1])
    if s.sum(): mids.append(np.median(areas[s])); means.append(dices[s].mean())
ax[1].plot(mids, means, "o-", ms=7, color="darkorange")
ax[1].set_xscale("log"); ax[1].set_xlabel("median tumour area in bin (log)")
ax[1].set_ylabel("mean Dice"); ax[1].set_title("binned by size quintile"); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

from scipy.stats import spearmanr
rho, p = spearmanr(areas, dices)
print(f"Spearman correlation between tumour size and Dice: rho={rho:.3f}, p={p:.2g}")
print(f"\nsmallest 20% of tumours -> mean Dice {dices[areas <= bins[1]].mean():.3f}")
print(f"largest 20% of tumours  -> mean Dice {dices[areas >= bins[4]].mean():.3f}")

In [ ]:
# TODO: the sharper version of the question.
#
# Dice punishes boundary error relative to tumour size. So compute, for each slice,
# the boundary error in ABSOLUTE terms -- how many pixels did we get wrong --
# and plot THAT against tumour size instead.
#
# If absolute error is roughly flat across sizes but Dice falls off a cliff,
# you have shown the model is equally good everywhere and the METRIC is what changes.
# That is a genuinely different claim, and a much stronger one.

wrong = np.array([np.logical_xor(pred[k], masks[val_idx[k]]).sum()
                  for k in range(len(val_idx)) if masks[val_idx[k]].sum() > 0])

# TODO: plot `wrong` against `areas`. What does it tell you that the Dice plot didn't?

### Track B — going further
- Re-run the whole thing on a validation set you deliberately stuffed with only large tumours,
  then only small ones. How far apart are the two "headline Dice" numbers for *the same model*?
- Look up **Hausdorff distance** and **surface Dice**, and argue which you'd report to a surgeon.
- If you find Dice is size-biased: propose a fairer way to summarise this model in one number,
  and defend it.

---
# Track C — Can tumour shape predict its DNA?

**Why anyone cares.** This is the actual question the original paper asked. Lower-grade gliomas get
sorted into molecular subtypes based on genomic sequencing — expensive, slow, and it needs tissue.
An MRI is none of those things. So: **is the genetic subtype written in the tumour's shape?**

If yes, even weakly, you could get a hint about a tumour's biology from an image alone.

**The experiment.** Take *your model's own predicted masks*, extract shape descriptors (area,
circularity, eccentricity, solidity, extent), then test whether those descriptors differ across
genomic clusters.

**Be careful, and be honest.** With ~110 patients, split across several clusters, you have very
little statistical power. You will probably find a weak signal or none. If you torture five features
against five clusters you will find something "significant" by chance — that is exactly the trap
this track is designed to teach you about.

In [ ]:
#@title Track C — extract shape features from YOUR model's predictions
import pandas as pd, warnings
warnings.filterwarnings("ignore")

model, overall = train_model(train_idx, val_idx, epochs=20, seed=0)
print(f"model Dice: {overall:.4f}")

# predict on EVERY patient (we need all of them for statistics, not just val)
all_idx = np.arange(len(images))
pred_all = predict_all(model, images, all_idx)

feat_pred = pd.DataFrame(patient_shape_features(pred_all, patient_ids))
feat_true = pd.DataFrame(patient_shape_features(masks,    patient_ids))
print(f"\nshape features for {len(feat_pred)} patients from predicted masks")
feat_pred.head()

In [ ]:
#@title Join to the genomic table
meta = get_meta()
feat_pred["patient_short"] = feat_pred["patient"].map(short_id)
df = feat_pred.merge(meta, on="patient_short", how="inner")
print(f"{len(df)} patients matched to genomic data")
print("available outcome columns:", [c for c in meta.columns if c != "patient_short"])
df.head()

In [ ]:
#@title Test one feature against one cluster
FEATURE = "circularity"        #@param ["area","perimeter","circularity","eccentricity","extent","solidity"]
OUTCOME = "RNASeqCluster"      #@param {type:"string"}

d = df[[FEATURE, OUTCOME]].dropna()
groups = [g[FEATURE].values for _, g in d.groupby(OUTCOME) if len(g) >= 3]

if len(groups) >= 2:
    from scipy.stats import kruskal
    stat, p = kruskal(*groups)
    plt.figure(figsize=(6,3.4))
    plt.boxplot(groups, labels=[f"cluster {int(k)}\n(n={len(g)})"
                                for (k, g) in d.groupby(OUTCOME) if len(g) >= 3])
    plt.ylabel(FEATURE); plt.title(f"{FEATURE} by {OUTCOME}   (Kruskal-Wallis p = {p:.3f})")
    plt.grid(axis="y", alpha=.3); plt.tight_layout(); plt.show()
    print(f"p = {p:.4f}")
    print("p < 0.05 means: if there were NO real difference, data this extreme would show up")
    print("less than 5% of the time. It does NOT mean you found something important.")
else:
    print("Not enough patients per group to test.")

In [ ]:
#@title The multiple-comparisons check — run this before you believe anything above
# TODO: you just tested ONE feature against ONE outcome. What if you test all of them?

feats = ["area","perimeter","circularity","eccentricity","extent","solidity"]
outcomes = [c for c in meta.columns if c not in ("patient_short",)]

from scipy.stats import kruskal
rows = []
for f in feats:
    for o in outcomes:
        d = df[[f,o]].dropna()
        gs = [g[f].values for _, g in d.groupby(o) if len(g) >= 3]
        if len(gs) >= 2:
            try: rows.append((f, o, kruskal(*gs)[1]))
            except Exception: pass

res = pd.DataFrame(rows, columns=["feature","outcome","p"]).sort_values("p")
print(res.head(10).to_string(index=False))
n = len(res)
print(f"\n{n} tests run. At p<0.05 you'd expect ~{0.05*n:.1f} 'hits' by pure chance.")
print(f"You got {(res.p < 0.05).sum()}.")
print(f"Bonferroni-corrected threshold: p < {0.05/n:.5f}. Survivors: {(res.p < 0.05/n).sum()}")
print("\nTODO: does anything survive? Write down your conclusion honestly, either way.")

### Track C — going further
- Repeat the whole analysis using the **radiologist's** masks (`feat_true`) instead of your model's.
  If the signal only appears with your model's masks, you have found a bug, not biology.
- Instead of the single largest slice, aggregate features over the whole 3D tumour volume.
- Try predicting the cluster with a small classifier and report cross-validated accuracy against
  the base rate of the commonest class. Does it beat "always guess the most common"?

---
# Present your work

Five minutes. Three slides. That's it.

**Slide 1 — the question.** What did you ask, and why would anyone care?

**Slide 2 — the plot.** One figure. Axes labelled. Say what it shows.

**Slide 3 — the answer, and the caveat.** What you found, in one sentence. Then: what would make you
doubt it? Every honest result comes with one.

> Nobody is grading you on getting a high Dice. You are being graded on whether you can be trusted
> about what your numbers mean. That distinction is the entire job.

In [ ]:
#@markdown ### My capstone
track = "A"  #@param ["A","B","C"]
my_question = ""  #@param {type:"string"}
what_i_found = ""  #@param {type:"string"}
what_would_make_me_doubt_it = ""  #@param {type:"string"}
print(f"Track {track}. Go present it.")